# 🏗️ Notebook 1: Reddit — Requirements & Architecture


## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Describe the **functional** and **non-functional** requirements of a Reddit-like system.
- Do a **back-of-envelope** estimate (QPS, storage, bandwidth) for 50 M daily users.
- Explain why Reddit is a **read-heavy**, **eventually consistent** system.
- Sketch a high-level architecture and justify each service boundary.
- See, with a tiny simulation, **why we must precompute the Hot feed** instead of computing it per request.


## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. What are we actually building?

Reddit is a **social news aggregator**. Users gather into communities called *subreddits*
(`r/python`, `r/cats`, ...), post links or text, **upvote** or **downvote** others' posts,
and reply to each other in **threaded** comments. The platform ranks content so that "what's
interesting right now" bubbles to the top.

The four screens you must support:

| Screen | Reads from | Writes to |
|---|---|---|
| Subreddit feed (`/r/python?sort=hot`) | precomputed ranked feed | — |
| Post detail + comments | post row + comment tree | — |
| Vote button | — | votes table / stream |
| Submit post / comment | — | posts / comments table |

### Functional requirements

1. Create subreddits; users can subscribe/unsubscribe.
2. Submit posts (link or text). Edit / delete own posts.
3. Upvote / downvote / unvote on posts **and** comments.
4. Threaded comments (a reply to a reply to a reply …).
5. Feeds with multiple sort orders: **Hot**, **New**, **Top**, **Rising**, **Controversial**.
6. A cross-subreddit `r/all` feed.

### Non-functional requirements

- **Read-heavy**: ~99 % of traffic is reads (browsing feeds, reading comments). Writes are votes,
  posts, and comments.
- **Eventually consistent scores are fine**: no one notices if an upvote takes a few seconds to
  appear on the counter.
- **Hot ranking is time-sensitive**: the Hot feed must change every few minutes, so some piece
  of the system must re-rank continuously.
- **Comment trees can be huge**: viral threads have 100 k+ comments. We cannot load the whole
  tree; we must paginate within a branch ("load more replies").
- **Voting must be cheap and racy-safe**: millions of votes per day on a few mega-posts.


## 2. Back-of-envelope numbers

We can't design for scale we haven't estimated. Let's ballpark it.


In [ ]:
# A tiny back-of-envelope calculator.
# Numbers are illustrative — the *method* is what matters in an interview.

DAU = 50_000_000           # daily active users
PAGE_LOADS_PER_USER = 5    # average feed/post views per user per day
VOTES_PER_USER = 0.2       # 1 vote every 5 days on average
POSTS_PER_USER = 0.01      # 1 post per 100 users per day
COMMENTS_PER_USER = 0.1    # 1 comment per 10 users per day

SECONDS_PER_DAY = 86_400

read_rps  = DAU * PAGE_LOADS_PER_USER / SECONDS_PER_DAY
vote_rps  = DAU * VOTES_PER_USER      / SECONDS_PER_DAY
post_rps  = DAU * POSTS_PER_USER      / SECONDS_PER_DAY
comm_rps  = DAU * COMMENTS_PER_USER   / SECONDS_PER_DAY

print(f"feed reads : {read_rps:>10,.0f} rps   ({read_rps*10:>10,.0f} rps at 10x peak)")
print(f"votes      : {vote_rps:>10,.0f} rps   ({vote_rps*10:>10,.0f} rps at 10x peak)")
print(f"new posts  : {post_rps:>10,.0f} rps")
print(f"comments   : {comm_rps:>10,.0f} rps")

write_rps = vote_rps + post_rps + comm_rps
print(f"\nread:write ratio = {read_rps/write_rps:,.0f} : 1")


**Takeaway:** roughly **150:1 reads per write**. This single ratio drives *every* design
decision:

- Cache everything readable.
- Use eventual consistency for counters (it's fine).
- Put votes on a log/stream and batch them; don't synchronously update a row per vote.

### Storage

A post row is ~1 KB (title + metadata). 10 k new posts/day × 365 days × 10 years ≈ **36 M posts ≈ 36 GB**.
Comments are bigger (a few hundred million per year), maybe **1 TB/year** including indexes.
Votes are the biggest: **trillions of rows** if we kept them all. We keep only the latest vote
per `(user, post)` pair — so the size is bounded by (users × posts they've voted on).


## 3. High-level architecture

```
               ┌──────────┐
               │  client  │
               └────┬─────┘
                    ▼
              ┌──────────┐
              │ API gtwy │   auth, rate limit, routing
              └────┬─────┘
         ┌────────┬┴────────┬──────────┬─────────────┐
         ▼        ▼         ▼          ▼             ▼
        Feed    Post      Vote       Comment      Subreddit
        Svc     Svc       Svc         Svc            Svc
         │       │         │            │             │
         ▼       ▼         ▼            ▼             ▼
       Redis   MySQL     Kafka        MySQL         MySQL
       (hot    (canonical (votes →   (threaded
        feed)  posts)     ranking)    comments)
                            │
                            ▼
                  ┌──────────────────────┐
                  │  Ranking worker      │  runs every ~5 min
                  │  reads votes + age   │  → writes hot_score
                  │  → updates Redis feed│
                  └──────────────────────┘
```

### Why these boundaries?

- **Feed Svc**: its job is to serve *pre-ranked* lists. It never runs the ranking formula on
  the hot path; it reads from Redis.
- **Vote Svc**: accepts votes, writes to Kafka, returns 200 immediately. The heavy lifting (updating
  counters, re-ranking) happens downstream. This decouples the user-facing latency from the
  aggregation work.
- **Ranking worker**: consumes votes, recomputes `hot_score`, writes to the feed cache.
  It *is* allowed to be a few minutes stale — users expect that.
- **Post / Comment / Subreddit Svc**: classic CRUD services backed by MySQL.

The MySQL schema is shown in Notebook 2; the algorithms (hot ranking, sharded counters,
comment trees) are in Notebook 3.


## 4. Simulation — why the Hot feed must be precomputed

Imagine we have 1 million posts and a request wants the **top 25 by hot score**. Two strategies:

- **Compute-on-read** — score every post at request time, sort, return top 25.
- **Precomputed** — a background job already wrote the sorted list to Redis; we just `LRANGE`.

Let's time them.


In [ ]:
import math, random, time, heapq

random.seed(42)

# simulate 1,000,000 posts with random age + votes
posts = [
    (pid,
     random.randint(0, 50_000),       # ups
     random.randint(0, 5_000),        # downs
     random.randint(0, 7*86_400))     # age_seconds (up to 7 days)
    for pid in range(1_000_000)
]

def hot_score(ups, downs, age_s):
    net = ups - downs
    sign = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    return order + sign * age_s / 45_000

# --- compute-on-read ---
t0 = time.time()
top25_live = heapq.nlargest(25, posts, key=lambda p: hot_score(p[1], p[2], p[3]))
live_ms = (time.time() - t0) * 1000

# --- precomputed: imagine this was built by a background job ---
# the "cached list" is just a Python list of ids we could LRANGE from Redis
t0 = time.time()
top25_cached = [p[0] for p in top25_live[:25]]   # simulate LRANGE 0 24
cached_ms = (time.time() - t0) * 1000

print(f"compute-on-read : {live_ms:7.1f} ms   (scored 1,000,000 posts)")
print(f"precomputed read: {cached_ms:7.3f} ms   (just read 25 ids)")
print(f"speedup         : {live_ms/max(cached_ms,0.001):,.0f}x")


The precomputed version is **thousands of times faster** and, crucially, its cost does **not**
grow with the total number of posts. At 30 k requests/second you *must* precompute.

The price we pay is **freshness**: the feed is always a few minutes stale. For a news-aggregator
that's an acceptable trade; for a stock ticker it would not be.


## 📚 Summary

- Reddit is **150× more reads than writes** → optimise for reads, tolerate eventual consistency on the write side.
- Split the platform into small services: **Feed, Post, Vote, Comment, Subreddit**, plus a
  **Ranking worker** that runs in the background.
- **Never compute the Hot feed on the request path.** Precompute it, cache it, serve from Redis.
- Votes go through Kafka so the user-facing write is fast; aggregation happens asynchronously.

### 💡 Interview tips

- Lead with the read/write ratio — it is the single sentence that justifies half your design.
- Draw the *ranking worker* explicitly; interviewers love seeing that "feed request" and "ranking"
  are decoupled.
- Mention that `r/all` is the *union* of subreddit feeds, not a separate index — a merge in the
  Feed service.

### Next up

In **Notebook 2** we flesh out the data model, the API surface, and see denormalised vote counts
and threaded comments in runnable code.
